<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week6/lesson23/Lesson23_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 23 — Exercises**
## Coupled ODEs and the Wilson–Cowan Model: Phase Planes, Jacobians and Oscillations

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 6 — Differential Equations\
**Lesson:** Lesson 23

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | Linear systems and their modes | 15 |
| 2 | Phase plane of the FitzHugh–Nagumo neuron | 15 |
| 3 | Linearisation: how local is it? | 15 |
| 4 | A decision circuit: fixed points and basins | 15 |
| 5 | The Wilson–Cowan model | 15 |
| 6 | Bifurcation intuition | 10 |
| 7 | Real data: is the first neuron linear? | 15 |
| **Total** | | **100** |
| Bonus | Noise-driven switching in a bistable circuit | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, fsolve, least_squares
from scipy.linalg import expm

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def jacobian_fd(F, x, h=1e-6):
    # PROVIDED: numerical Jacobian J[i, j] = dF_i/dx_j by centred differences.
    x = np.asarray(x, float); J = np.zeros((len(x), len(x)))
    for j in range(len(x)):
        e = np.zeros(len(x)); e[j] = h
        J[:, j] = (np.asarray(F(x + e)) - np.asarray(F(x - e))) / (2 * h)
    return J

print("Setup complete.")

---
## Exercise 1 — Linear Systems and Their Modes (15 points)

For $\dot{\mathbf x} = A\mathbf x$, the solution is $\mathbf x(t) = e^{At}\mathbf x_0 = P e^{\Lambda t}P^{-1}\mathbf x_0$ when $A = P\Lambda P^{-1}$.

In [ ]:
# 1.1 (6 points): For A1 below (units 1/ms) and x0 = (1, 0):
# (a) store the eigenvalues lam1 (np.linalg.eig), the decay time tau_dec1 = 1 / |Re(lambda)| (ms) and the oscillation period period1 = 2 pi / |Im(lambda)| (ms);
# (b) compute x(t) on t1 three ways: the modal formula P exp(Lambda t) P^{-1} x0 (take the real part), expm(A t) x0, and forward Euler with dt = 0.01 ms.
#     Store err_modal1 = max |modal - expm| and err_euler1 = max |Euler - expm| (over both components and all times).

A1 = np.array([[-0.05, -0.30], [0.30, -0.05]])                                # (2, 2) 1/ms
x0_1 = np.array([1.0, 0.0])
t1 = np.arange(0, 100.001, 0.01)                                              # (10001,) ms

# YOUR CODE HERE
lam1 = tau_dec1 = period1 = err_modal1 = err_euler1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.1
assert all(v is not None for v in (lam1, tau_dec1, period1, err_modal1, err_euler1)), "Set all five results."
assert np.isclose(tau_dec1, 20.0) and np.isclose(period1, 2 * np.pi / 0.3), "Re = -0.05, Im = +-0.3."
assert err_modal1 < 1e-10 and err_euler1 < 5e-3, "Three methods agree."
print("Self-check passed.")

In [ ]:
# 1.2 (5 points): A resonant membrane is often written as a second-order ODE, u'' + 2 zeta omega u' + omega^2 u = 0. With x = (u, u'), write it as
# x' = M x. For omega = 0.05 rad/ms (about 8 Hz) and each zeta in zeta_list2, store the eigenvalues (eig_list1, list of arrays), the trace and
# determinant (trdet1, list of (tr, det)), and the classification class1 (list of 'stable node' / 'stable focus' / ...), using tr^2 vs 4 det.
# Write it as a function classify(J) returning 'saddle', 'centre', 'stable/unstable node' or 'stable/unstable focus': you will reuse it.

zeta_list2 = [0.2, 0.9, 2.0]
omega1 = 0.05                                                                 # rad/ms

# YOUR CODE HERE
eig_list1 = trdet1 = class1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert all(v is not None for v in (eig_list1, trdet1, class1)), "Set eig_list1, trdet1 and class1."
assert class1 == ['stable focus', 'stable focus', 'stable node'], "Underdamped (zeta < 1): focus; overdamped: node."
assert all(np.isclose(d, omega1**2) for _, d in trdet1), "det M = omega^2 for every zeta."
print("Self-check passed.")

**1.3 (4 points, written response).** (a) Why do the eigenvalues of a real $2\times2$ matrix either both real or a complex-conjugate pair? (b) For $\zeta = 0.2$, how many oscillation periods occur during one decay time? What does that mean for how "ringing" the response looks? (c) Why does Euler's error in 1.1 stay small here, and when would a step of 0.01 ms be too large?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 2 — Phase Plane of the FitzHugh–Nagumo Neuron (15 points)

The FitzHugh–Nagumo model is a two-variable caricature of Hodgkin–Huxley (dimensionless units):
$$\dot v = v - \frac{v^3}{3} - w + I, \qquad \tau\,\dot w = v + a - bw, \qquad a = 0.7,\ b = 0.8,\ \tau = 12.5.$$
$v$ plays the role of voltage (fast, with a cubic nonlinearity) and $w$ of a slow recovery variable.

In [ ]:
# 2.1 (6 points): (a) Write the nullclines as functions: w_vnull(v, I) = v - v**3 / 3 + I and w_wnull(v) = (v + a) / b.
# (b) For each I in I_list2, find the fixed point v* with brentq on [-3, 3] (the nullclines cross once here), and store fp2 = {I: (v*, w*)}.
# (c) Store J_an2 = {I: analytical Jacobian [[1 - v*^2, -1], [1/tau, -b/tau]]}, check it against jacobian_fd, and store class2 = {I: classification}.

a2, b2, tau2 = 0.7, 0.8, 12.5
I_list2 = [0.0, 0.5]

# YOUR CODE HERE
fp2 = J_an2 = class2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.1
assert all(v is not None for v in (fp2, J_an2, class2)), "Set fp2, J_an2 and class2."
assert class2[0.0] == 'stable focus' and class2[0.5] == 'unstable focus', "Rest is stable at I = 0; at I = 0.5 the fixed point is unstable."
assert all(abs(fhn(np.array(fp2[I_]), I_)).max() < 1e-9 for I_ in I_list2), "Fixed points are zeros of both rates."
print("Self-check passed.")

In [ ]:
# 2.2 (5 points): Simulate with solve_ivp (rtol 1e-8, max_step 0.1) from (v, w) = (-1.2, -0.6) for t in [0, 400]:
# (a) at I = 0.5: store the amplitude of v after t = 200 (amp2 = max - min) and the period (period2, mean interval between peaks of v);
# (b) at I = 0 with a brief kick: start at the fixed point plus (dv, 0) for dv in [0.2, 0.5, 1.0], and store the maximum of v reached (vmax2, shape (3,)).
#     Plot the phase plane for I = 0.5 with both nullclines and the trajectory.

# YOUR CODE HERE
amp2 = period2 = vmax2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert all(v is not None for v in (amp2, period2, vmax2)), "Set amp2, period2 and vmax2."
assert amp2 > 3 and 30 < period2 < 50, "A large, regular oscillation (repetitive spiking)."
assert vmax2[0] < 0 and vmax2[-1] > 1.5, "Small kicks decay; a large kick produces a full 'spike' (excitability)."
print("Self-check passed.")

**2.3 (4 points, written response).** (a) Explain, using the nullclines, why a small kick from rest decays but a larger one produces a large excursion before returning (excitability). (b) Why does the fixed point lose stability as $I$ increases? Refer to the Jacobian entry $1 - v^{*2}$. (c) In what sense is FitzHugh–Nagumo a better neuron model than the LIF of Lesson 22, and in what sense is it worse?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 3 — Linearisation: How Local Is It? (15 points)

Near a fixed point, $\dot{\boldsymbol\delta} \approx J\boldsymbol\delta$, so $\boldsymbol\delta(t) \approx e^{Jt}\boldsymbol\delta_0$. We test how large $\boldsymbol\delta_0$ can be.

In [ ]:
# 3.1 (6 points): Use FitzHugh-Nagumo at I = 0 (from Exercise 2). For kicks delta0 = (d, 0) with d in d_list3, simulate the nonlinear system for 60 time units
# (solve_ivp, rtol 1e-9, t_eval every 0.1) and the linearisation expm(J t) delta0. Store rel_err3 (shape (5,)): the maximum over time of
# |delta_nonlinear - delta_linear| (Euclidean norm) divided by d.

d_list3 = [0.01, 0.05, 0.1, 0.2, 0.4]

# YOUR CODE HERE
rel_err3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert rel_err3 is not None, "Set rel_err3."
assert rel_err3[0] < 0.02 and np.all(np.diff(rel_err3) > 0) and rel_err3[-1] > 10 * rel_err3[0], "Linearisation is accurate for small kicks, and the relative error grows with the kick."
print("Self-check passed.")

In [ ]:
# 3.2 (5 points): Predicting the response frequency. For the FitzHugh-Nagumo fixed point at I = 0, store omega_lin3 = |Im(lambda)| of J and the
# predicted period T_lin3 = 2 pi / omega_lin3. From the simulation with d = 0.01, measure the interval between the first two maxima of v (T_sim3).

# YOUR CODE HERE
omega_lin3 = T_lin3 = T_sim3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert all(v is not None for v in (omega_lin3, T_lin3, T_sim3)), "Set omega_lin3, T_lin3 and T_sim3."
assert abs(T_sim3 - T_lin3) / T_lin3 < 0.02, "The Jacobian predicts the frequency of small oscillations."
print("Self-check passed.")

**3.3 (4 points, written response).** (a) Why does the relative error of the linearisation grow with the kick size? Relate it to the Taylor expansion. (b) A neuroscientist injects small sinusoidal currents to measure a neuron's resonance frequency. Why must the currents be small, and how does 3.2 justify reading the frequency off the Jacobian? (c) When can the linearisation give the wrong qualitative answer about stability?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — A Decision Circuit: Fixed Points and Basins (15 points)

Two populations with self-excitation $w_s$ and cross-inhibition $w_c$ (the teaching notebook's model):
$\tau\dot r_1 = -r_1 + F(w_s r_1 - w_c r_2 + I_1)$, $\tau\dot r_2 = -r_2 + F(w_s r_2 - w_c r_1 + I_2)$, $F(u) = 1/(1 + e^{-(u - 0.5)/0.1})$, $\tau = 10$ ms.

In [ ]:
# 4.1 (6 points): With w_s = 0.6 and I1 = I2 = 0.45, for each cross-inhibition w_c in wc_list4 find ALL fixed points (fsolve from a 6 x 6 grid of
# starts in [0, 1]^2; keep distinct ones with residual < 1e-9) and classify each with jacobian_fd and classify. Store n_fp4 = {w_c: number of fixed points}
# and n_stable4 = {w_c: number of stable fixed points}.

def F4(u):
    return 1 / (1 + np.exp(-(u - 0.5) / 0.1))
def dec4(x, w_s=0.6, w_c=1.0, I1=0.45, I2=0.45, tau=10.0):
    return np.array([(-x[0] + F4(w_s * x[0] - w_c * x[1] + I1)) / tau, (-x[1] + F4(w_s * x[1] - w_c * x[0] + I2)) / tau])
wc_list4 = [0.0, 0.3, 0.6, 1.0]

# YOUR CODE HERE
n_fp4 = n_stable4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert n_fp4 is not None and n_stable4 is not None, "Set n_fp4 and n_stable4."
assert n_stable4[0.0] == 1 and n_stable4[1.0] == 2 and n_fp4[1.0] == 3, "Without competition one stable state; with strong competition two (a decision)."
print("Self-check passed.")

In [ ]:
# 4.2 (5 points): Bias and basins. With w_c = 1.0, vary I1 in I1_list4 (I2 = 0.45). For each, integrate a 41 x 41 grid of initial conditions in [0, 1]^2
# for 300 ms with vectorised Euler (dt = 0.1 ms) and store frac1_4 (shape (4,)): the fraction of starts for which r1 > r2 + 0.1 at the end.
# Also store frac_sym4: the fraction from the symmetric start r1 = r2 = 0.1 that choice 1 wins under each bias (1.0 or 0.0, shape (4,)).

I1_list4 = [0.45, 0.46, 0.48, 0.52]

# YOUR CODE HERE
frac1_4 = frac_sym4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert frac1_4 is not None and frac_sym4 is not None, "Set frac1_4 and frac_sym4."
assert np.all(np.diff(frac1_4) > 0) and frac1_4[-1] > 0.55 and frac_sym4[-1] == 1.0, "A larger input to population 1 enlarges its basin."
print("Self-check passed.")

**4.3 (4 points, written response).** (a) In 4.1, explain how increasing cross-inhibition turns one stable state into two. What kind of fixed point appears between them? (b) Why is the symmetric start always "undecided" with equal inputs in a deterministic model, and what in a real brain would break the tie? (c) Relate the basins to a psychometric function: what would you predict for choices with small input differences?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — The Wilson–Cowan Model (15 points)

$\tau_E\dot r_E = -r_E + F_E(w_{EE}r_E - w_{EI}r_I + P)$, $\tau_I\dot r_I = -r_I + F_I(w_{IE}r_E - w_{II}r_I + Q)$, $F(x) = 1/(1 + e^{-a(x - \theta)}) - 1/(1 + e^{a\theta})$, with the teaching notebook's parameters.

In [ ]:
# 5.1 (6 points): Set A (bistable). (a) Starting from the low state (0, 0), apply P = 1.0 for 30 ms, then P = 0 for 470 ms (solve_ivp, max_step 0.5);
# store rE_end5 = rE at 500 ms. (b) Then, from that state, apply an inhibitory pulse to the excitatory population
# (P = -2.0 for 30 ms, then P = 0 for 470 ms); store rE_end5b. (c) Store the number of fixed points of Set A with P = 0 (n_fp5) and their classifications (cls5, list).

def F_wc(x, a, theta):
    return 1 / (1 + np.exp(-a * (x - theta))) - 1 / (1 + np.exp(a * theta))
WC = dict(tE=10.0, tI=20.0, aE=1.2, thE=2.8, aI=1.0, thI=4.0, Q=0.0)
SET_A = dict(WC, wEE=9.0, wEI=4.0, wIE=13.0, wII=11.0, P=0.0)
SET_B = dict(WC, wEE=6.4, wEI=4.8, wIE=6.0, wII=1.2, P=0.9)
def wc(x, p, P=None):
    P = p['P'] if P is None else P
    return np.array([(-x[0] + F_wc(p['wEE'] * x[0] - p['wEI'] * x[1] + P, p['aE'], p['thE'])) / p['tE'],
                     (-x[1] + F_wc(p['wIE'] * x[0] - p['wII'] * x[1] + p['Q'], p['aI'], p['thI'])) / p['tI']])

# YOUR CODE HERE
rE_end5 = rE_end5b = n_fp5 = cls5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert all(v is not None for v in (rE_end5, rE_end5b, n_fp5, cls5)), "Set all four results."
assert rE_end5 > 0.9 and rE_end5b < 0.01, "A brief pulse switches the circuit on; a brief negative pulse switches it off."
assert n_fp5 == 3 and cls5[1] == 'saddle', "Low and high stable states separated by a saddle."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Set B (oscillating). For tau_I in tauI_list5, simulate 3000 ms from (0.1, 0.05) and measure, after 2000 ms, the oscillation frequency
# (Hz, from the mean interval between peaks of rE; nan if the range of rE is below 0.01) and amplitude (max - min of rE). Store freq5 and amp5 (shape (4,)).

tauI_list5 = [5.0, 10.0, 20.0, 40.0]                                           # ms

# YOUR CODE HERE
freq5 = amp5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert freq5 is not None and amp5 is not None, "Set freq5 and amp5."
assert np.isnan(freq5[0]) or amp5[0] < amp5[2], "Fast inhibition damps or weakens the oscillation."
ok = ~np.isnan(freq5)
assert ok.sum() >= 2 and np.all(np.diff(freq5[ok]) < 0), "Slower inhibition, slower rhythm."
print("Self-check passed.")

**5.3 (4 points, written response).** (a) Describe one cycle of the Set B oscillation in words, in terms of excitation and inhibition. (b) Why does slowing inhibition slow the rhythm, and why can very fast inhibition abolish it? (c) Gamma oscillations (30–80 Hz) are thought to arise from this mechanism. Which parameters would you change to move this model into the gamma range?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Bifurcation Intuition (10 points)

In [ ]:
# 6.1 (6 points): (a) For the linear sag model J(g) = [[-1/tau_m, -1/tau_m], [g/tau_w, -1/tau_w]] with tau_m = 30 ms, tau_w = 150 ms, find with brentq the coupling
# g_star6 at which tr^2 = 4 det (node -> focus), and compare with the formula g* = tau_m tau_w tr^2 / 4 - 1 (g_formula6).
# (b) For Wilson-Cowan Set B, find with brentq on P in [0.9, 1.1] the drive P_hopf6 at which the largest real part of the eigenvalues of the (single) fixed
# point is zero. Use fsolve from (0.6, 0.3) to find the fixed point for each P.

# YOUR CODE HERE
g_star6 = g_formula6 = P_hopf6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert all(v is not None for v in (g_star6, g_formula6, P_hopf6)), "Set g_star6, g_formula6 and P_hopf6."
assert np.isclose(g_star6, g_formula6) and np.isclose(g_star6, 0.8), "tr^2 = 4 det gives g* = 0.8."
assert 0.95 < P_hopf6 < 1.02, "The Hopf point is near P = 1."
print("Self-check passed.")

**6.2 (4 points, written response).** (a) The sag model crosses from node to focus at $g^*$, but it never becomes unstable for any $g > 0$. Why not? (b) What changes qualitatively in the Wilson–Cowan model at $P_{\text{Hopf}}$, and what happens to the oscillation amplitude as $P$ approaches it from below? (c) Why might a neuroscientist care where bifurcation points are (give one example)?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Is the First Neuron Linear? (15 points)

**Data.** Lesson 21's first neuron (`18711001.abf`, pyABF sample files, MIT licence; cell type undocumented): `patch_18711001_steps.npz`, 30 sweeps at 10 kHz. Each sweep holds −100 pA from 23.4 to 323.4 ms. Lesson 21 found that its recovery after the step was slower than its charging (34 against 17.5 ms).
- `V7` (30, 15000): membrane potential, mV. `I7` (30, 15000): injected current, pA. `t7` (15000,): time, s.

**A key fact.** For any **linear** system starting at rest, the response to switching a constant input **off** after a long step is the mirror image of the response to switching it **on**: if $r(t)$ is the response to a step at $t = 0$, a step from 0 to $T$ gives $r(t) - r(t - T)$ for $t > T$. So for a linear system the on and off time courses must have the same shape once the step is long enough.

In [ ]:
# PROVIDED: load the recording.
import os, urllib.request
DATA_FILE7 = 'patch_18711001_steps.npz'
for cand in [DATA_FILE7, os.path.join('..', '..', 'data', DATA_FILE7)]:
    if os.path.exists(cand):
        path7 = cand; break
else:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/peter-adepoju/math-neuro-ml/e8996ac071a2a16c48d5b544ec3470816eecf49f/data/' + DATA_FILE7, DATA_FILE7)
    path7 = DATA_FILE7
d7 = np.load(path7)
V7 = d7['v_cmV'] / 100.0                                                       # (30, 15000) mV
I7 = d7['i_pA'].astype(float)                                                  # (30, 15000) pA
fs7 = float(d7['fs_hz']); t7 = np.arange(V7.shape[1]) / fs7                    # (15000,) s
on7, off7 = np.where(np.diff(I7[0]) != 0)[0][:2] + 1                            # samples where -100 pA starts and ends
Vm7 = V7.mean(axis=0); E_L7 = Vm7[:on7].mean()
u7 = Vm7[:int(0.8 * fs7)] - E_L7                                               # (8000,) mV, 0-800 ms
I_seq7 = I7[0, :int(0.8 * fs7)]                                                # (8000,) pA
print(f"E_L = {E_L7:.2f} mV;  step from {t7[on7] * 1e3:.1f} to {t7[off7] * 1e3:.1f} ms")

In [ ]:
# 7.1 (7 points): (a) Fit the two-variable sag model of the teaching notebook (tau_m, R, tau_w, g) to the ON response (samples on7 to off7 of u7, input
# -100 pA), using the exact piecewise-constant update and least_squares (x0 = [20, 0.2, 100, 0.2], bounds ([1, 0.01, 5, 0], [200, 1, 2000, 5])).
# Store p7 (shape (4,)), rmse_on7 (mV), the Jacobian eigenvalues ev7 and the classification cls7.
# (b) Simulate the fitted model over the whole 0-800 ms with I_seq7 and store rmse_off7, the RMSE over the 150 ms after the step ends.
# (c) Store rmse_off_1var7 for the one-variable model with tau = 17.45 ms and R = 0.222 mV/pA over the same window.

# YOUR CODE HERE
p7 = rmse_on7 = ev7 = cls7 = rmse_off7 = rmse_off_1var7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert all(v is not None for v in (p7, rmse_on7, ev7, cls7, rmse_off7, rmse_off_1var7)), "Set all six results."
assert rmse_on7 < 0.3 and cls7 == 'stable node', "The ON response is fitted well by a stable node."
assert rmse_off7 > 5 * rmse_on7, "But the model fails on the OFF response."
print("Self-check passed.")

In [ ]:
# 7.2 (4 points): A model-free linearity test. Normalise the ON response as f_on(s) = u(on + s) / u_end and the OFF response as f_off(s) = 1 - u(off + s) / u_end,
# where u_end = mean of u7 over the last 20 ms of the step. For a linear system that has (nearly) reached steady state, f_on and f_off coincide.
# Store the half-times (ms) at which each first exceeds 0.5 (t_half_on7, t_half_off7) and their ratio half_ratio7. Also store the jump in V within
# the first 0.2 ms after the step ends (jump7, mV).

# YOUR CODE HERE
t_half_on7 = t_half_off7 = half_ratio7 = jump7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert all(v is not None for v in (t_half_on7, t_half_off7, half_ratio7, jump7)), "Set all four results."
assert 0.5 < jump7 < 3 and half_ratio7 > 1.1, "A fast jump at offset, and a slower OFF than ON time course."
print("Self-check passed.")

**7.3 (4 points, written response).** (a) Why do on and off responses of a linear system have the same shape? Use superposition (Lesson 21, Exercise 6). (b) What do 7.1 and 7.2 together tell you about this neuron, and why can **no** linear model, with any number of variables, fit both responses? (c) Name two possible causes of the asymmetry, and an experiment or analysis that could tell them apart.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — Noise-Driven Switching in a Bistable Circuit (extra credit)

Add Gaussian noise to the excitatory equation of Wilson–Cowan Set A, using Euler–Maruyama (Lesson 22): $r_E \leftarrow r_E + \Delta t\,\dot r_E + \sigma\sqrt{\Delta t}\,\eta$.
1. For $\sigma$ in [0.02, 0.04, 0.06, 0.08] (per $\sqrt{\text{ms}}$), simulate 20 s starting in the low state ($\Delta t = 0.1$ ms, `np.random.default_rng(7)`) and count switches between the low ($r_E < 0.2$) and high ($r_E > 0.8$) states.
2. How does the switching rate depend on $\sigma$? Compare with the idea of a potential barrier (the saddle).
3. Relate this to up and down states in cortex.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, a legend where appropriate, and no background grid.
- Every fixed point you report comes with its Jacobian eigenvalues and a classification; every simulation states its method and time step or tolerance.
- You checked at least one numerical result against an analytical one (eigenvalues, nullcline intersections, or a linearised prediction).
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.